# Loop Detection in Comparison Graph

Computes the set of comparisons whose removal makes the comparison graph
acyclic: for each cyclic SCC, cut the most suspect edge on one remaining cycle
at a time - largest positive score gap first (loser scored above winner),
oldest vote as tiebreak - until no cycle is left. Scores are public stored
ratings (win probability against an average image), chosen by the
cut-stability experiment.

Flow: load proxies via CrystalGraph - build the cut graph - run the cut phase
with explicit rules - inspect in the viewer - remove (DRY_RUN / MAX_REMOVALS)
- verify.

In [ ]:
from comfyui_image_scorer.core.observability.logger import (
    ModuleLogger,
    get_logger,
    configure_package_logging,
)

# 20 = INFO; configure_package_logging sets the __main__ logger level
configure_package_logging(20)
logger: ModuleLogger = get_logger(__name__)

# 0 removes the whole cut list in one pass
MAX_REMOVALS: int = 0

SHOW_N_ELEMENTS: int = 0

DRY_RUN: bool = False


## 1. Load Data

In [ ]:
import time

from comfyui_image_scorer.adapters.cli.deps import CLIDeps, build_cli_deps
from comfyui_image_scorer.application.services.graph_service import CrystalGraph
from comfyui_image_scorer.domain.graph.chain_manager import ChainManager
from comfyui_image_scorer.domain.graph.link_proxy import LinkProxy
from comfyui_image_scorer.domain.graph.node_proxy import NodeProxy

t0: float = time.perf_counter()
deps: CLIDeps = build_cli_deps()
graph: CrystalGraph = deps.graph
graph.rebuild_from_database()

nodes: list[NodeProxy] = graph.get_all_nodes()
links: list[LinkProxy] = graph.get_all_links()
# experiment-only access to the chain manager the graph already owns
cm: ChainManager = graph._chain

logger.info(
    "Loaded %d comparisons across %d images",
    len(links),
    len(nodes),
    start_timer=t0,
)


## 2. Build Graph and Run Tarjan SCC

In [ ]:
from comfyui_image_scorer.domain.comparison.algorithm.loop_detection import (
    CutGraph,
    build_cut_graph,
)

cut_graph: CutGraph = build_cut_graph(cm, links)
sizes: list[int] = [len(members) for _, members in cut_graph.cyclic_sccs]
logger.info(
    "Tarjan: %d cyclic SCCs covering %d nodes, top sizes %s",
    len(sizes),
    len(cut_graph.cyclic_members),
    sizes[:20],
)


## 3. Compute the Cut List

In [ ]:
from comfyui_image_scorer.domain.comparison.algorithm.loop_detection import (
    PassResult,
    run_cut_phase,
)

result: PassResult = run_cut_phase(cut_graph, nodes, "public", "gap_ts")


## 4. Cut List Viewer

In [ ]:
from pathlib import Path

import matplotlib.image as mpimg
import matplotlib.pyplot as plt

from comfyui_image_scorer.domain.comparison.algorithm.loop_detection import (
    CutEntry,
    proxy_scores,
)
from comfyui_image_scorer.domain.graph.node_proxy import NodeProxy
from comfyui_image_scorer.infrastructure.persistence.path_handler import find_image_path

_SIDE_STYLES: tuple[tuple[str, str], tuple[str, str]] = (
    ("WINNER", "tab:green"),
    ("LOSER", "tab:red"),
)


def build_cut_figure(
    entries: list[CutEntry], nodes: list[NodeProxy], show_top: int, max_removals: int
) -> None:
    shown: list[CutEntry] = entries[:show_top] if show_top > 0 else entries
    if not shown:
        logger.warning("Cut list is empty, nothing to display")
        return
    apply_count: int = (
        len(entries) if max_removals <= 0 else min(max_removals, len(entries))
    )
    public_scores, _ = proxy_scores(nodes, "public")
    db_scores: dict[str, float] = {node.filename: node.score for node in nodes}
    fig, axes = plt.subplots(
        len(shown), 2, figsize=(10, 4.8 * len(shown)), squeeze=False
    )
    for row_i, cut in enumerate(shown):
        link = cut.link
        winner_fn: str = link.winner
        loser_fn: str = link.loser
        apply_tag: str = "  [WILL REMOVE]" if row_i < apply_count else "  [LATER PASS]"
        header: str = (
            f"Cut {row_i + 1}/{len(entries)}{apply_tag}\n"
            f"{winner_fn[:40]} -> {loser_fn[:40]}\n"
            f"gap={cut.gap:+.3f}  id={link.id}  ts={link.timestamp}\n"
            f"SCC {cut.scc_id} (size {cut.scc_size})  cycle length {cut.cycle_length}"
        )
        for col, filename in enumerate((winner_fn, loser_fn)):
            side_label, side_color = _SIDE_STYLES[col]
            ax = axes[row_i][col]
            path: Path | None = find_image_path(filename)
            if path is None:
                ax.text(
                    0.5,
                    0.5,
                    f"not found:\n{filename}",
                    ha="center",
                    va="center",
                    fontsize=8,
                    wrap=True,
                )
            else:
                ax.imshow(mpimg.imread(path))
            title: str = (
                f"{side_label} | public={public_scores.get(filename, 0.0):.3f} (db={db_scores.get(filename, 0.0):.3f})\n{filename}"
            )
            if col == 0:
                title = f"{header}\n{title}"
            ax.set_title(title, fontsize=7.5, color=side_color)
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_edgecolor(side_color)
                spine.set_linewidth(3)
    fig.tight_layout()
    fig.subplots_adjust(hspace=0.5, wspace=0.06)
    plt.show()


build_cut_figure(result.entries, nodes, SHOW_N_ELEMENTS, MAX_REMOVALS)


## 5. Removal

In [ ]:
from comfyui_image_scorer.domain.comparison.algorithm.loop_detection import (
    CutEntry,
    cycles_after_removal,
)
from comfyui_image_scorer.infrastructure.persistence.comparisons_repository import (
    remove_comparison,
)

applied: list[CutEntry] = (
    result.entries if MAX_REMOVALS <= 0 else result.entries[:MAX_REMOVALS]
)

if not result.entries:
    logger.warning("Nothing to remove - the cut list is empty")
elif not result.complete:
    logger.error("Refusing removal: cut set incomplete")
elif DRY_RUN:
    logger.info(
        "Dry run: %d of %d cuts would be applied (MAX_REMOVALS=%d)",
        len(applied),
        len(result.entries),
        MAX_REMOVALS,
    )
else:
    removed: int = 0
    ok_pairs: set[tuple[str, str]] = set()
    for cut in applied:
        if remove_comparison(filename_a=cut.canon[0], filename_b=cut.canon[1]):
            removed += 1
            ok_pairs.add(cut.canon)
        else:
            logger.warning("No comparison rows found for pair %s", cut.canon)

    post_cyclic: list[list[str]] = cycles_after_removal(cm, ok_pairs)
    if not post_cyclic:
        logger.info("Graph clean: 0 cycles after removing %d pairs", removed)
    elif len(applied) < len(result.entries):
        logger.info(
            "Staged pass: applied %d of %d cuts; %d SCC(s) still cyclic (top sizes %s); rerun the notebook to continue",
            removed,
            len(result.entries),
            len(post_cyclic),
            sorted((len(m) for m in post_cyclic), reverse=True)[:20],
        )
    else:
        logger.error(
            "Unexpected: %d cyclic SCC(s) remain after the complete cut set (top sizes %s)",
            len(post_cyclic),
            sorted((len(m) for m in post_cyclic), reverse=True)[:20],
        )
    logger.info("Run 'scorer.py database sync' so the changes persist")
